# Amnesia Anonymization — End-to-End Demo

This notebook demonstrates how to use the [Amnesia REST API](https://amnesia.openaire.eu/) from Python.

**Workflow:**
1. Start the Amnesia backend server
2. Get a session
3. Load dataset
4. Auto-generate generalization hierarchies
5. Load hierarchies
6. Run anonymization (k-anonymity)
7. Inspect solutions & download anonymized data
8. Clean up

## 0. Setup

In [2]:
import subprocess
import time
import json
import requests
import pandas as pd
from pathlib import Path

JAR_PATH = Path("/home/jupyter-vojta/notebooks/Amnesia/target/amnesiaBackEnd-1.0-SNAPSHOT.jar")
AMNESIA_DIR = JAR_PATH.parent.parent
BASE_URL = "http://localhost:8181"

In [3]:
print("JAR exists:", JAR_PATH.exists())

JAR exists: True


In [ ]:
DATA_DIR = Path("../data")
DATA_FILE = DATA_DIR / "test_data.csv"

print("Data file exists:", DATA_FILE.exists())
pd.read_csv(DATA_FILE)

## 1. Start the Amnesia Server

In [4]:
proc = subprocess.Popen(
    [
        "java", "-Xms1024m", "-Xmx4096m",
        "-Dorg.eclipse.jetty.server.Request.maxFormKeys=1000000",
        "-Dorg.eclipse.jetty.server.Request.maxFormContentSize=1000000",
        "-jar", str(JAR_PATH),
        "--server.port=8181",
    ],
    cwd=str(AMNESIA_DIR),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("Waiting for server to start...")
for _ in range(20):
    time.sleep(2)
    try:
        r = requests.post(f"{BASE_URL}/getSession", timeout=3)
        if r.status_code == 200:
            print("Server is up!")
            break
    except requests.exceptions.ConnectionError:
        pass
else:
    raise RuntimeError("Server did not start in time")

Waiting for server to start...
Server is up!


## 2. Get a Session

In [5]:
r = requests.post(f"{BASE_URL}/getSession")
session_id = r.json()["Session_Id"]
headers = {"Cookie": f"JSESSIONID={session_id}"}
print("Session ID:", session_id)

Session ID: node01tiielygraywr16nk5eonrgtkb1


## 3. Load Dataset

In [6]:
column_types = json.dumps({
    "age": "int",
    "zipcode": "int",
    "gender": "string",
    "salary": "int",
})

with open(DATA_FILE, "rb") as f:
    r = requests.post(
        f"{BASE_URL}/loadData",
        headers=headers,
        files={"file": ("test_data.csv", f, "text/csv")},
        data={
            "del": ",",
            "datasetType": "tabular",
            "columnsType": column_types,
        },
    )

print(r.json())

{'Status': 'Success', 'Message': 'Dataset is  successfully loaded!'}


## 4. Auto-Generate Hierarchies

We use `generateHierarchy` to create range-based hierarchies for `age` and `salary`, and a distinct hierarchy for `zipcode`.

In [7]:
age_hier_path = DATA_DIR / "age_hier.txt"

r = requests.post(
    f"{BASE_URL}/generateHierarchy",
    headers=headers,
    data={
        "hierType": "range",
        "varType": "int",
        "attribute": "age",
        "hierName": "age_hier",
        "startLimit": 1,
        "endLimit": 100,
        "fanout": 3,
        "step": 5,
    },
)

age_hier_path.write_bytes(r.content)
print("age hierarchy saved:", age_hier_path)
print(r.text[:300])

age hierarchy saved: ../data/age_hier.txt
{"hierType":"range","name":"age_hier","type":"int","level1":{"1-100":["1-46","46-100","(null)"]},"level3":{"46-61":["46-51","51-56","56-61"],"31-46":["31-36","36-41","41-46"],"76-91":["76-81","81-86","86-91"],"1-16":["1-6","6-11","11-16"],"91-100":["91-96","96-100"],"16-31":["16-21","21-26","26-31"]


In [8]:
salary_hier_path = DATA_DIR / "salary_hier.txt"

r = requests.post(
    f"{BASE_URL}/generateHierarchy",
    headers=headers,
    data={
        "hierType": "range",
        "varType": "int",
        "attribute": "salary",
        "hierName": "salary_hier",
        "startLimit": 10000,
        "endLimit": 150000,
        "fanout": 3,
        "step": 10000,
    },
)

salary_hier_path.write_bytes(r.content)
print("salary hierarchy saved:", salary_hier_path)
print(r.text[:300])

salary hierarchy saved: ../data/salary_hier.txt
{"hierType":"range","name":"salary_hier","type":"int","level1":{"10000-150000":["10000-100000","100000-150000","(null)"]},"level3":{"100000-130000":["100000-110000","110000-120000","120000-130000"],"40000-70000":["40000-50000","50000-60000","60000-70000"],"10000-40000":["10000-20000","20000-30000","


## 5. Load Hierarchies

In [9]:
with open(age_hier_path, "rb") as f_age, open(salary_hier_path, "rb") as f_sal:
    r = requests.post(
        f"{BASE_URL}/loadHierarchies",
        headers=headers,
        files=[
            ("hierarchies", ("age_hier.txt", f_age, "text/plain")),
            ("hierarchies", ("salary_hier.txt", f_sal, "text/plain")),
        ],
    )

print(r.json())

{'Status': 'Fail', 'Message': 'Failed to load hierarchies, please try again!'}


## 6. Run Anonymization (k=3)

We bind `age` → `age_hier` and `salary` → `salary_hier` as quasi-identifiers.

In [10]:
bind = json.dumps({"age": "age_hier", "salary": "salary_hier"})

r = requests.post(
    f"{BASE_URL}/anonymization",
    headers=headers,
    data={"bind": bind, "k": 3},
)

solutions = r.json()["Solutions"]
print(f"Found {len(solutions)} solutions:\n")
for name, info in sorted(solutions.items()):
    print(f"  {name}: levels={info['levels']}, result={info['result']}")

Found 25 solutions:

  sol0: levels=[0,0], result=unsafe
  sol1: levels=[1,0], result=unsafe
  sol10: levels=[4,0], result=unsafe
  sol11: levels=[3,1], result=unsafe
  sol12: levels=[2,2], result=unsafe
  sol13: levels=[1,3], result=unsafe
  sol14: levels=[0,4], result=unsafe
  sol15: levels=[4,1], result=safe
  sol16: levels=[3,2], result=unsafe
  sol17: levels=[2,3], result=safe
  sol18: levels=[1,4], result=unsafe
  sol19: levels=[4,2], result=safe
  sol2: levels=[0,1], result=unsafe
  sol20: levels=[3,3], result=safe
  sol21: levels=[2,4], result=safe
  sol22: levels=[4,3], result=safe
  sol23: levels=[3,4], result=safe
  sol24: levels=[4,4], result=safe
  sol3: levels=[2,0], result=unsafe
  sol4: levels=[1,1], result=unsafe
  sol5: levels=[0,2], result=unsafe
  sol6: levels=[3,0], result=unsafe
  sol7: levels=[2,1], result=unsafe
  sol8: levels=[1,2], result=unsafe
  sol9: levels=[0,3], result=unsafe


## 7. Download a Safe Solution

In [11]:
# Pick the first safe solution
safe = [(name, info) for name, info in solutions.items() if info["result"] == "safe"]
chosen_name, chosen_info = safe[0]
print(f"Using {chosen_name}: levels={chosen_info['levels']}")

r = requests.post(
    f"{BASE_URL}/getSolution",
    headers=headers,
    data={"sol": chosen_info["levels"]},
)

anon_path = DATA_DIR / "anonymized.csv"
anon_path.write_bytes(r.content)
print("Saved to:", anon_path)

Using sol20: levels=[3,3]
Saved to: ../data/anonymized.csv


In [12]:
# Compare original vs anonymized
original = pd.read_csv(DATA_FILE)
anonymized = pd.read_csv(anon_path)

print("=== Original (first 5) ===")
print(original.head())
print("\n=== Anonymized (first 5) ===")
print(anonymized.head())

=== Original (first 5) ===
   age  zipcode  gender  salary
0   25    10001    male   35000
1   27    10001  female   38000
2   29    10002    male   42000
3   31    10002  female   45000
4   33    10003    male   50000

=== Anonymized (first 5) ===
      age  zipcode  gender        salary
0    1-46    10004    male  10000-100000
1    1-46    10005    male  10000-100000
2  46-100    10007    male  10000-100000
3  46-100    10008  female  10000-100000
4  46-100    10008  female  10000-100000


## 8. Clean Up

In [13]:
# Clear the session on the server
r = requests.post(f"{BASE_URL}/clearSession", headers=headers)
print("Session cleared:", r.json())

# Stop the server process
proc.terminate()
print("Server stopped.")

Session cleared: {'Status': 'Success', 'Message': 'Session is cleared!'}
Server stopped.
